## Part 0 - Setup

Run this cell first, in every notebook. It fetches the course repository into the Colab session and moves into the `notebooks/` folder, so that the `../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory. Start Jupyter from inside `notebooks/` and the paths work the same way.

If it prints `data ok: True`, you are set.


In [ ]:
# --- SETUP: run this first ---  [lares-setup-v1]
# works in Colab and locally, safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/unizg-fer-lares/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))


In [ ]:
import numpy as np, pandas as pd

# same plot styling as in 1a
import seaborn as sns
sns.set_theme(style="whitegrid")
import matplotlib.pyplot as plt
tex_fonts = {
    "font.family": "serif",
    # Use 26pt font in plots
    "axes.labelsize": 20,
    "font.size": 20,
    "figure.titlesize": 20,
    # Make the legend/label fonts a little smaller
    "legend.title_fontsize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18
}

plt.rcParams.update(tex_fonts)


In [ ]:
# The sub-folder ``supervised_learning`` sits one level deeper than the
# shared ``data`` folder, so we add another ``../`` when needed.
DATA_DIR = "../data" if os.path.isdir("../data") else "../../data"
print("data folder:", DATA_DIR)


# Introduction to Classification: K-Nearest Neighbors

**Student exercise - notebook `2_KNN_Classification`**

We classify Iris flowers by asking their nearest neighbours to vote, and see how the number of neighbours changes the decision boundary.


## Part 1 - Load the Iris data

The same train and test files as in `2a_Classification_Iris`. To draw decision boundaries we keep only two features: petal length and petal width.


In [ ]:
X_train_full = pd.read_csv(f"{DATA_DIR}/iris/X_train.csv", index_col="Id")
X_test_full = pd.read_csv(f"{DATA_DIR}/iris/X_test.csv", index_col="Id")
y_train = pd.read_csv(f"{DATA_DIR}/iris/y_train.csv", index_col="Id").squeeze()
y_test = pd.read_csv(f"{DATA_DIR}/iris/y_test.csv", index_col="Id").squeeze()

feature_names = ["PetalLengthCm", "PetalWidthCm"]
X_train = X_train_full[feature_names]
X_test = X_test_full[feature_names]

species = {0: "Iris-setosa", 1: "Iris-versicolor", 2: "Iris-virginica"}
print("train flowers:", len(X_train), "| test flowers:", len(X_test))
X_train.head()


## Part 2 - Look at the classes

Each dot is a flower, coloured by its true species.


In [ ]:
train_view = X_train.copy()
train_view["Species"] = y_train.map(species)

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=train_view, x="PetalLengthCm", y="PetalWidthCm", hue="Species", s=90, ax=ax)
ax.set_title("Iris petals - three species in different regions")
plt.tight_layout()
plt.show()


## Part 3 - Scale the features

KNN measures **straight-line distance** between points, so features with big numbers dominate. Standardising each column (subtract mean, divide by standard deviation) puts them on the same footing.

We fit the scaler on the training data only.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("first 3 scaled train rows:")
print(pd.DataFrame(X_train_scaled, columns=feature_names).head(3))


## Part 4 - Fit a first KNN model

`KNeighborsClassifier(n_neighbors=5)` predicts by looking at the 5 nearest training flowers and taking the majority vote.

We also report three metrics:

* **Accuracy** - the fraction of flowers classified correctly.
* **Precision** - when the model says "this is species X", how often is it right?
* **Recall** - among all true species X flowers, how many did the model catch?

For 3 classes we ask for the weighted average of the per-class scores.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# TODO 1: create KNN with 5 neighbours and fit it on the scaled train data
knn5 = ...
...

# TODO 2: predict the test labels
y_pred = ...

# TODO 3: print weighted accuracy, precision, and recall
...


## Part 5 - Helper: draw the decision boundary

To see what a KNN model actually learned, we:

1. cover the plot with a fine grid of points,
2. ask the model to classify every grid point,
3. paint the background with the predicted colour.

The helper does that once so we can reuse it below.


In [ ]:
def plot_decision_boundary(ax, model, X_scaled, y_labels, title):
    x_min = X_scaled[:, 0].min() - 0.5
    x_max = X_scaled[:, 0].max() + 0.5
    y_min = X_scaled[:, 1].min() - 0.5
    y_max = X_scaled[:, 1].max() + 0.5

    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]
    grid_pred = model.predict(grid_points).reshape(grid_x.shape)

    ax.contourf(grid_x, grid_y, grid_pred, alpha=0.25,
                levels=[-0.5, 0.5, 1.5, 2.5], cmap="viridis")
    ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_labels, cmap="viridis",
               edgecolor="white", s=55)
    ax.set_title(title)
    ax.set_xlabel("Petal length (scaled)")
    ax.set_ylabel("Petal width (scaled)")


## Part 6 - K=1 vs. K=15

* With `K=1` every training flower rules its tiny area, so the boundary is jagged and can memorise noise.
* With `K=15` fifteen neighbours vote, so the boundary is smoother.


In [ ]:
# TODO 4: fit two KNN models, one with K=1 and one with K=15
knn1 = ...
knn15 = ...

# TODO 5: draw both decision boundaries side by side with plot_decision_boundary
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
...
plt.tight_layout()
plt.show()


## Part 7 - How do we pick K in practice?

We must not use the final test set to choose `K`. Instead we cut a **validation slice** out of the training data, try many values of `K`, and see which one wins on the slice.


In [ ]:
from sklearn.model_selection import train_test_split

# TODO 6: split the training data into a fit set and a validation set
X_fit, X_val, y_fit, y_val = ...

k_values = list(range(1, 21))
val_accuracy = []
train_accuracy = []

for k in k_values:
    # TODO 7: fit a KNN with this k, record its train and validation accuracy
    ...

best_k = k_values[int(np.argmax(val_accuracy))]
print("best K on the validation slice:", best_k)


### Accuracy curve

Look for the "elbow": the point where more neighbours stops helping.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, train_accuracy, marker="o", label="train accuracy")
ax.plot(k_values, val_accuracy, marker="s", label="validation accuracy")
ax.axvline(best_k, color="red", linestyle="--", label=f"best K = {best_k}")
ax.set_xlabel("K (number of neighbours)")
ax.set_ylabel("Accuracy")
ax.set_xticks(k_values)
ax.set_title("Elbow curve for K")
ax.legend()
plt.tight_layout()
plt.show()


## Part 8 - Final score with the chosen K

Now we retrain on all the training data with the best K and check the test set.


In [ ]:
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)

y_pred_final = final_knn.predict(X_test_scaled)
print(f"test accuracy  : {accuracy_score(y_test, y_pred_final):.3f}")
print(f"test precision : {precision_score(y_test, y_pred_final, average='weighted'):.3f}")
print(f"test recall    : {recall_score(y_test, y_pred_final, average='weighted'):.3f}")


### Conclusions - overall:
*   KNN classifies a sample by majority vote of its nearest training neighbours.
*   Standardising features keeps the distance calculation fair.
*   Very small K memorises noise; larger K smooths the boundary.
*   A validation slice is used to pick K without touching the test set.

_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - basic_  
_Notebook: 2_KNN_Classification_  

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_
